In [ ]:
from pathlib import Path
import json
import html

# ============================================================
# CONFIGURATION
# ============================================================

PROJECT_ROOT = Path.cwd()

PROCESSED_DATA_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

JSON_INPUT_FILE = (
    PROCESSED_DATA_DIR
    / "steam_games_2022_2025_compact.json"
)

APP_OUTPUT_DIR = (
    PROJECT_ROOT
    / "steam-game-suggester"
)

APP_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

HTML_OUTPUT_FILE = (
    APP_OUTPUT_DIR
    / "steam_game_suggester.html"
)

# ============================================================
# LOAD AND VALIDATE DATA
# ============================================================

with open(
    JSON_INPUT_FILE,
    "r",
    encoding="utf-8"
) as file:
    games_data = json.load(file)

if not isinstance(games_data, list):
    raise TypeError(
        "Expected the compact JSON to contain a list of game records."
    )

if len(games_data) == 0:
    raise ValueError(
        "The compact JSON contains no games."
    )

required_fields = {
    "appid",
    "name",
    "release_year",
    "price",
    "total_reviews",
    "review_score",
    "estimated_revenue",
    "revenue_rank_in_year",
    "qualifying_games_in_year",
    "revenue_percentile_in_year",
}

available_fields = set()

for game in games_data[:100]:
    available_fields.update(game.keys())

missing_fields = required_fields - available_fields

if missing_fields:
    raise ValueError(
        "The JSON is missing required fields: "
        + ", ".join(sorted(missing_fields))
    )

# Prevent a game description from accidentally closing the script tag
embedded_json = json.dumps(
    games_data,
    ensure_ascii=False,
    separators=(",", ":")
).replace(
    "</script>",
    "<\\/script>"
)

years = sorted({
    int(game["release_year"])
    for game in games_data
})

maximum_revenue = max(
    float(game.get("estimated_revenue", 0) or 0)
    for game in games_data
)

maximum_price = max(
    float(game.get("price", 0) or 0)
    for game in games_data
)

maximum_rank = max(
    int(game.get("revenue_rank_in_year", 1) or 1)
    for game in games_data
)

all_tags = sorted({
    str(tag).strip()
    for game in games_data
    for tag in (
        game.get("tags", [])
        if isinstance(game.get("tags", []), list)
        else (
            game.get("tags", {}).keys()
            if isinstance(game.get("tags", {}), dict)
            else []
        )
    )
    if str(tag).strip()
})

year_checkbox_html = "\n".join(
    f"""
    <label class="year-option">
        <input
            type="checkbox"
            class="year-checkbox"
            value="{year}"
            checked
        >
        <span>{year}</span>
    </label>
    """
    for year in years
)

tag_options_html = "\n".join(
    f'<option value="{html.escape(tag, quote=True)}"></option>'
    for tag in all_tags
)

tag_buttons_html = "\n".join(
    (
        f'<button class="available-tag" '
        f'type="button" '
        f'data-tag="{html.escape(tag, quote=True)}">'
        f'{html.escape(tag)}'
        f'</button>'
    )
    for tag in all_tags
)

# ============================================================
# BUILD HTML
# ============================================================
game_lookup_options_html = "\n".join(
    f'<option value="{html.escape(str(game["name"]), quote=True)}">'
    f'AppID {int(game["appid"])}</option>'
    for game in games_data
)

html_document = f"""<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">

    <meta
        name="viewport"
        content="width=device-width, initial-scale=1.0"
    >

    <title>Steam Indie Case Study Suggester</title>

    <style>
        :root {{
            color-scheme: dark;

            --background: #080b12;
            --panel: #111722;
            --panel-soft: #151d2b;
            --panel-hover: #1a2434;

            --border: #263247;
            --border-soft: #1e293b;

            --text: #f4f7fb;
            --muted: #98a6ba;
            --subtle: #6f7e93;

            --accent: #69c0ff;
            --accent-strong: #25a7ff;
            --accent-soft: rgba(37, 167, 255, 0.13);

            --green: #67d391;
            --green-soft: rgba(103, 211, 145, 0.14);

            --orange: #ffbd66;
            --danger: #ff7b7b;

            --shadow:
                0 24px 60px rgba(0, 0, 0, 0.34);

            --radius-large: 24px;
            --radius-medium: 16px;
            --radius-small: 10px;
        }}

        * {{
            box-sizing: border-box;
        }}

        html {{
            min-height: 100%;
        }}

        body {{
            margin: 0;
            min-height: 100vh;

            font-family:
                Inter,
                ui-sans-serif,
                system-ui,
                -apple-system,
                BlinkMacSystemFont,
                "Segoe UI",
                sans-serif;

            color: var(--text);

            background:
                radial-gradient(
                    circle at 12% 12%,
                    rgba(37, 167, 255, 0.10),
                    transparent 28%
                ),
                radial-gradient(
                    circle at 90% 84%,
                    rgba(103, 211, 145, 0.08),
                    transparent 30%
                ),
                var(--background);
        }}

        button,
        input {{
            font: inherit;
        }}

        button {{
            cursor: pointer;
        }}

        .app-shell {{
            width: min(1480px, calc(100% - 32px));
            margin: 0 auto;
            padding: 34px 0 54px;
        }}

        .topbar {{
            display: flex;
            align-items: flex-start;
            justify-content: space-between;
            gap: 24px;
            margin-bottom: 26px;
        }}

        .eyebrow {{
            margin: 0 0 8px;

            color: var(--accent);
            font-size: 0.78rem;
            font-weight: 800;
            letter-spacing: 0.15em;
            text-transform: uppercase;
        }}

        h1 {{
            margin: 0;
            max-width: 800px;

            font-size: clamp(2rem, 4vw, 3.6rem);
            line-height: 0.98;
            letter-spacing: -0.055em;
        }}

        .intro {{
            max-width: 720px;
            margin: 16px 0 0;

            color: var(--muted);
            font-size: 1rem;
            line-height: 1.65;
        }}

        .dataset-badge {{
            flex: 0 0 auto;
            padding: 12px 16px;

            color: var(--green);
            font-size: 0.88rem;
            font-weight: 750;

            background: var(--green-soft);
            border: 1px solid rgba(103, 211, 145, 0.24);
            border-radius: 999px;
        }}

        .layout {{
            display: grid;
            grid-template-columns: minmax(290px, 365px) minmax(0, 1fr);
            gap: 24px;
            align-items: start;
        }}

        .panel {{
            background:
                linear-gradient(
                    145deg,
                    rgba(255, 255, 255, 0.022),
                    rgba(255, 255, 255, 0)
                ),
                var(--panel);

            border: 1px solid var(--border-soft);
            border-radius: var(--radius-large);
            box-shadow: var(--shadow);
        }}

        .filters {{
            position: sticky;
            top: 18px;

            max-height: calc(100vh - 36px);
            padding: 22px;
            overflow-y: auto;
        }}

        .panel-heading {{
            display: flex;
            align-items: center;
            justify-content: space-between;
            gap: 12px;

            margin-bottom: 18px;
        }}

        .panel-heading h2 {{
            margin: 0;
            font-size: 1.05rem;
        }}

        .match-count {{
            color: var(--accent);
            font-size: 0.84rem;
            font-weight: 750;
        }}

        .filter-group {{
            padding: 18px 0;
            border-top: 1px solid var(--border-soft);
        }}

        .filter-group:first-of-type {{
            padding-top: 4px;
            border-top: 0;
        }}

        .filter-label {{
            display: block;
            margin-bottom: 10px;

            color: var(--text);
            font-size: 0.86rem;
            font-weight: 720;
        }}

        .filter-help {{
            display: block;
            margin-top: 7px;

            color: var(--subtle);
            font-size: 0.74rem;
            line-height: 1.45;
        }}

        .year-grid {{
            display: grid;
            grid-template-columns: repeat(2, minmax(0, 1fr));
            gap: 8px;
        }}

        .year-option {{
            position: relative;
        }}

        .year-option input {{
            position: absolute;
            opacity: 0;
            pointer-events: none;
        }}

        .year-option span {{
            display: flex;
            align-items: center;
            justify-content: center;

            min-height: 40px;

            color: var(--muted);
            font-weight: 720;

            background: var(--panel-soft);
            border: 1px solid var(--border);
            border-radius: 10px;

            transition:
                border-color 140ms ease,
                background 140ms ease,
                color 140ms ease;
        }}

        .year-option input:checked + span {{
            color: var(--accent);
            background: var(--accent-soft);
            border-color: rgba(37, 167, 255, 0.52);
        }}

        .input-pair {{
            display: grid;
            grid-template-columns: 1fr 1fr;
            gap: 10px;
        }}

        .input-wrapper {{
            min-width: 0;
        }}

        .input-caption {{
            display: block;
            margin-bottom: 5px;

            color: var(--subtle);
            font-size: 0.72rem;
        }}

        input[type="number"],
        input[type="text"] {{
            width: 100%;
            min-width: 0;
            padding: 11px 12px;

            color: var(--text);

            background: #0d131e;
            border: 1px solid var(--border);
            border-radius: 10px;
            outline: none;

            transition:
                border-color 140ms ease,
                box-shadow 140ms ease;
        }}

        input:focus {{
            border-color: var(--accent-strong);
            box-shadow:
                0 0 0 3px rgba(37, 167, 255, 0.12);
        }}

        .toggle-row {{
            display: flex;
            align-items: center;
            justify-content: space-between;
            gap: 14px;
        }}

        .toggle-copy strong {{
            display: block;
            font-size: 0.85rem;
        }}

        .toggle-copy span {{
            display: block;
            margin-top: 3px;

            color: var(--subtle);
            font-size: 0.74rem;
        }}

        .switch {{
            position: relative;
            flex: 0 0 auto;

            width: 48px;
            height: 27px;
        }}

        .switch input {{
            position: absolute;
            opacity: 0;
        }}

        .switch-track {{
            position: absolute;
            inset: 0;

            background: #273246;
            border-radius: 999px;

            transition: background 160ms ease;
        }}

        .switch-track::after {{
            content: "";

            position: absolute;
            top: 4px;
            left: 4px;

            width: 19px;
            height: 19px;

            background: white;
            border-radius: 50%;

            transition: transform 160ms ease;
        }}

        .switch input:checked + .switch-track {{
            background: var(--accent-strong);
        }}

        .switch input:checked + .switch-track::after {{
            transform: translateX(21px);
        }}

        .tag-browser {{
            margin-top: 12px;
        }}

        .tag-browser summary {{
            cursor: pointer;

            color: var(--accent);
            font-size: 0.78rem;
            font-weight: 750;
        }}

        .tag-search-wrapper {{
            margin-top: 12px;
        }}

        .available-tag-list {{
            display: flex;
            flex-wrap: wrap;
            gap: 7px;

            max-height: 260px;
            margin-top: 10px;
            padding: 10px;

            overflow-y: auto;

            background: #0d131e;
            border: 1px solid var(--border);
            border-radius: 12px;
        }}

        .available-tag {{
            padding: 6px 9px;

            color: var(--muted);
            font-size: 0.72rem;

            background: var(--panel-soft);
            border: 1px solid var(--border);
            border-radius: 999px;

            transition:
                color 120ms ease,
                border-color 120ms ease,
                background 120ms ease;
        }}

        .available-tag:hover {{
            color: var(--accent);
            border-color: rgba(37, 167, 255, 0.55);
            background: var(--accent-soft);
        }}

        .available-tag.selected {{
            color: var(--accent);
            border-color: rgba(37, 167, 255, 0.55);
            background: var(--accent-soft);
        }}

        .selected-tag-summary {{
            display: flex;
            flex-wrap: wrap;
            gap: 6px;
            margin-top: 10px;
        }}

        .selected-filter-tag {{
            display: inline-flex;
            align-items: center;
            gap: 5px;

            padding: 5px 8px;

            color: var(--accent);
            font-size: 0.7rem;
            font-weight: 700;

            background: var(--accent-soft);
            border: 1px solid rgba(37, 167, 255, 0.38);
            border-radius: 999px;
        }}

        .selected-filter-tag button {{
            padding: 0;

            color: var(--accent);
            line-height: 1;

            background: transparent;
            border: 0;
        }}

        .button-stack {{
            display: grid;
            gap: 10px;
            margin-top: 8px;
        }}

        .primary-button,
        .secondary-button,
        .ghost-button,
        .store-button {{
            min-height: 46px;
            padding: 0 18px;

            font-weight: 800;
            border-radius: 12px;

            transition:
                transform 120ms ease,
                filter 120ms ease,
                border-color 120ms ease,
                background 120ms ease;
        }}

        .primary-button {{
            color: #03111b;

            background:
                linear-gradient(
                    135deg,
                    #7bd0ff,
                    #37abff
                );

            border: 0;
        }}

        .primary-button:hover {{
            transform: translateY(-1px);
            filter: brightness(1.06);
        }}

        .primary-button:disabled {{
            cursor: not-allowed;
            opacity: 0.42;
            transform: none;
        }}

        .secondary-button {{
            color: var(--text);
            background: var(--panel-soft);
            border: 1px solid var(--border);
        }}

        .secondary-button:hover {{
            border-color: #40516c;
            background: var(--panel-hover);
        }}

        .ghost-button {{
            min-height: 38px;
            padding: 0 13px;

            color: var(--muted);
            font-size: 0.78rem;

            background: transparent;
            border: 1px solid var(--border);
        }}

        .ghost-button:hover {{
            color: var(--text);
            border-color: #40516c;
        }}

        .result-column {{
            min-width: 0;
        }}

        .empty-state {{
            min-height: 660px;
            padding: 60px;

            display: grid;
            place-items: center;

            text-align: center;
        }}

        .empty-icon {{
            width: 76px;
            height: 76px;
            margin: 0 auto 20px;

            display: grid;
            place-items: center;

            color: var(--accent);
            font-size: 2rem;

            background: var(--accent-soft);
            border: 1px solid rgba(37, 167, 255, 0.24);
            border-radius: 22px;
        }}

        .empty-state h2 {{
            margin: 0;
            font-size: 1.55rem;
        }}

        .empty-state p {{
            max-width: 520px;
            margin: 12px auto 0;

            color: var(--muted);
            line-height: 1.65;
        }}

        .case-study {{
            overflow: hidden;
        }}

        .hidden {{
            display: none !important;
        }}

        .hero {{
            position: relative;
            min-height: 440px;
            overflow: hidden;

            background: #090e17;
        }}

        .hero-image {{
            display: block;

            width: 100%;
            height: 440px;

            object-fit: cover;

            transition:
                opacity 180ms ease,
                filter 180ms ease;
        }}

        .hero-image.loading {{
            opacity: 0.32;
            filter: blur(6px);
        }}

        .hero-gradient {{
            position: absolute;
            inset: 0;

            background:
                linear-gradient(
                    to top,
                    rgba(7, 10, 17, 0.96) 0%,
                    rgba(7, 10, 17, 0.45) 43%,
                    rgba(7, 10, 17, 0.08) 75%
                );
        }}

        .hero-content {{
            position: absolute;
            left: 32px;
            right: 32px;
            bottom: 28px;
        }}

        .case-label {{
            display: inline-flex;
            align-items: center;
            gap: 8px;

            margin-bottom: 12px;
            padding: 7px 11px;

            color: var(--green);
            font-size: 0.73rem;
            font-weight: 800;
            letter-spacing: 0.08em;
            text-transform: uppercase;

            background: rgba(7, 10, 17, 0.72);
            border: 1px solid rgba(103, 211, 145, 0.32);
            border-radius: 999px;
            backdrop-filter: blur(10px);
        }}

        .game-title {{
            max-width: 1000px;
            margin: 0;

            font-size: clamp(2rem, 5vw, 4.3rem);
            line-height: 0.98;
            letter-spacing: -0.055em;
            text-wrap: balance;
        }}

        .hero-meta {{
            display: flex;
            flex-wrap: wrap;
            gap: 8px;

            margin-top: 15px;
        }}

        .hero-chip {{
            padding: 7px 10px;

            color: #dce6f5;
            font-size: 0.78rem;
            font-weight: 700;

            background: rgba(7, 10, 17, 0.72);
            border: 1px solid rgba(255, 255, 255, 0.13);
            border-radius: 999px;
            backdrop-filter: blur(10px);
        }}

        .case-content {{
            padding: 28px;
        }}

        .metric-grid {{
            display: grid;
            grid-template-columns: repeat(3, minmax(0, 1fr));
            gap: 12px;
        }}

        .metric {{
            min-width: 0;
            padding: 16px;

            background: var(--panel-soft);
            border: 1px solid var(--border-soft);
            border-radius: 14px;
        }}

        .metric-label {{
            display: block;
            margin-bottom: 7px;

            color: var(--subtle);
            font-size: 0.72rem;
            font-weight: 700;
            text-transform: uppercase;
            letter-spacing: 0.07em;
        }}

        .metric-value {{
            display: block;

            overflow: hidden;
            text-overflow: ellipsis;

            color: var(--text);
            font-size: clamp(1rem, 2vw, 1.35rem);
            font-weight: 820;
            white-space: nowrap;
        }}

        .metric-value.accent {{
            color: var(--accent);
        }}

        .metric-value.green {{
            color: var(--green);
        }}

        .details-grid {{
            display: grid;
            grid-template-columns: minmax(0, 1.55fr) minmax(260px, 0.75fr);
            gap: 26px;

            margin-top: 28px;
        }}

        .section-title {{
            margin: 0 0 12px;
            font-size: 0.9rem;
            letter-spacing: 0.01em;
        }}

        .description {{
            margin: 0;

            color: #cad4e3;
            font-size: 1rem;
            line-height: 1.72;
        }}

        .tag-list {{
            display: flex;
            flex-wrap: wrap;
            gap: 8px;
        }}

        .tag {{
            padding: 7px 10px;

            color: #bdc9da;
            font-size: 0.76rem;
            font-weight: 650;

            background: #0d141f;
            border: 1px solid var(--border);
            border-radius: 999px;
        }}

        .rank-card {{
            padding: 18px;

            background:
                linear-gradient(
                    145deg,
                    rgba(37, 167, 255, 0.10),
                    rgba(37, 167, 255, 0.025)
                );

            border: 1px solid rgba(37, 167, 255, 0.24);
            border-radius: 16px;
        }}

        .rank-card-label {{
            color: var(--muted);
            font-size: 0.77rem;
        }}

        .rank-card-value {{
            display: block;
            margin-top: 7px;

            color: var(--accent);
            font-size: 2rem;
            font-weight: 860;
            letter-spacing: -0.04em;
        }}

        .rank-card-note {{
            display: block;
            margin-top: 5px;

            color: var(--subtle);
            font-size: 0.74rem;
            line-height: 1.45;
        }}

        .actions {{
            display: flex;
            flex-wrap: wrap;
            gap: 10px;

            margin-top: 28px;
            padding-top: 24px;

            border-top: 1px solid var(--border-soft);
        }}

        .store-button {{
            display: inline-flex;
            align-items: center;
            justify-content: center;
            gap: 8px;

            color: #04111a;
            text-decoration: none;

            background: var(--green);
            border: 0;
        }}

        .store-button:hover {{
            transform: translateY(-1px);
            filter: brightness(1.05);
        }}

        .history-panel {{
            margin-top: 18px;
            padding: 20px;
        }}

        .history-list {{
            display: grid;
            gap: 8px;
        }}

        .history-item {{
            display: grid;
            grid-template-columns: 40px minmax(0, 1fr) auto;
            gap: 12px;
            align-items: center;

            padding: 10px 12px;

            background: var(--panel-soft);
            border: 1px solid var(--border-soft);
            border-radius: 12px;
        }}

        .history-rank {{
            color: var(--accent);
            font-size: 0.78rem;
            font-weight: 800;
        }}

        .history-name {{
            overflow: hidden;
            text-overflow: ellipsis;

            font-size: 0.84rem;
            font-weight: 700;
            white-space: nowrap;
        }}

        .history-revenue {{
            color: var(--muted);
            font-size: 0.76rem;
        }}

        .toast {{
            position: fixed;
            left: 50%;
            bottom: 28px;
            z-index: 100;

            max-width: calc(100% - 32px);
            padding: 11px 16px;

            color: var(--text);
            font-size: 0.84rem;
            font-weight: 700;

            background: #162131;
            border: 1px solid #33445e;
            border-radius: 999px;
            box-shadow: var(--shadow);

            opacity: 0;
            pointer-events: none;
            transform: translate(-50%, 12px);

            transition:
                opacity 160ms ease,
                transform 160ms ease;
        }}

        .toast.visible {{
            opacity: 1;
            transform: translate(-50%, 0);
        }}

        @media (max-width: 1080px) {{
            .layout {{
                grid-template-columns: 1fr;
            }}

            .filters {{
                position: static;
                max-height: none;
            }}

            .metric-grid {{
                grid-template-columns: repeat(3, minmax(0, 1fr));
            }}
        }}

        @media (max-width: 720px) {{
            .app-shell {{
                width: min(100% - 20px, 1480px);
                padding-top: 20px;
            }}

            .topbar {{
                display: block;
            }}

            .dataset-badge {{
                display: inline-flex;
                margin-top: 18px;
            }}

            .hero {{
                min-height: 360px;
            }}

            .hero-image {{
                height: 360px;
            }}

            .hero-content {{
                left: 20px;
                right: 20px;
                bottom: 20px;
            }}

            .case-content {{
                padding: 20px;
            }}

            .metric-grid {{
                grid-template-columns: repeat(2, minmax(0, 1fr));
            }}

            .details-grid {{
                grid-template-columns: 1fr;
            }}

            .empty-state {{
                min-height: 500px;
                padding: 30px 18px;
            }}

            .input-pair {{
                grid-template-columns: 1fr;
            }}
        }}
    </style>
</head>

<body>
    <main class="app-shell">
        <header class="topbar">
            <div>
                <p class="eyebrow">
                    Steam market study tool
                </p>

                <h1>
                    Indie Case Study Suggester
                </h1>

                <p class="intro">
                    Filter the recent Steam market and draw a random game
                    to study its positioning, pricing, commercial outcome,
                    tags, artwork, review score, and store pitch.
                </p>
            </div>

            <div class="dataset-badge">
                {len(games_data):,} qualifying games
            </div>
        </header>

        <div class="layout">
            <aside class="panel filters">
                <div class="panel-heading">
                    <h2>Study filters</h2>
                    <section class="filter-group">
                        <label
                            class="filter-label"
                            for="gameLookup"
                        >
                            Find a specific game
                        </label>

                        <input
                            id="gameLookup"
                            type="text"
                            list="gameLookupOptions"
                            placeholder="Type game name or AppID"
                            autocomplete="off"
                        >

                        <datalist id="gameLookupOptions">
                            {game_lookup_options_html}
                        </datalist>

                        <button
                            class="secondary-button"
                            id="loadGameButton"
                            type="button"
                            style="width:100%; margin-top:10px;"
                        >
                            Load game report
                        </button>

                        <span class="filter-help">
                            Search ignores the current filters and loads the exact game report.
                        </span>
                    </section>
                    <span
                        class="match-count"
                        id="matchCount"
                    >
                        Calculating…
                    </span>
                </div>

                <section class="filter-group">
                    <label class="filter-label">
                        Release years
                    </label>

                    <div class="year-grid">
                        {year_checkbox_html}
                    </div>
                </section>

                <section class="filter-group">
                    <label class="filter-label">
                        Estimated revenue
                    </label>

                    <div class="input-pair">
                        <label class="input-wrapper">
                            <span class="input-caption">Minimum</span>

                            <input
                                id="minRevenue"
                                type="number"
                                min="0"
                                step="10000"
                                value="150000"
                            >
                        </label>

                        <label class="input-wrapper">
                            <span class="input-caption">Maximum</span>

                            <input
                                id="maxRevenue"
                                type="number"
                                min="0"
                                step="10000"
                                value="800000"
                            >
                        </label>
                    </div>

                    <span class="filter-help">
                        Estimated as total reviews × 45 × listed price.
                    </span>
                </section>

                <section class="filter-group">
                    <label class="filter-label">
                        Listed price
                    </label>

                    <div class="input-pair">
                        <label class="input-wrapper">
                            <span class="input-caption">Minimum</span>

                            <input
                                id="minPrice"
                                type="number"
                                min="0"
                                step="1"
                                value="0"
                            >
                        </label>

                        <label class="input-wrapper">
                            <span class="input-caption">Maximum</span>

                            <input
                                id="maxPrice"
                                type="number"
                                min="0"
                                step="1"
                                value="{max(100, int(maximum_price) + 1)}"
                            >
                        </label>
                    </div>
                </section>

                <section class="filter-group">
                    <label class="filter-label">
                        Revenue rank within release year
                    </label>

                    <div class="input-pair">
                        <label class="input-wrapper">
                            <span class="input-caption">Best rank</span>

                            <input
                                id="minRank"
                                type="number"
                                min="1"
                                step="1"
                                value="1"
                            >
                        </label>

                        <label class="input-wrapper">
                            <span class="input-caption">Worst rank</span>

                            <input
                                id="maxRank"
                                type="number"
                                min="1"
                                step="1"
                                value="{maximum_rank}"
                            >
                        </label>
                    </div>
                </section>

                <section class="filter-group">
                    <label class="filter-label">
                        Revenue percentile
                    </label>

                    <div class="input-pair">
                        <label class="input-wrapper">
                            <span class="input-caption">Minimum percentile</span>

                            <input
                                id="minPercentile"
                                type="number"
                                min="0"
                                max="100"
                                step="1"
                                value="0"
                            >
                        </label>

                        <label class="input-wrapper">
                            <span class="input-caption">Maximum percentile</span>

                            <input
                                id="maxPercentile"
                                type="number"
                                min="0"
                                max="100"
                                step="1"
                                value="100"
                            >
                        </label>
                    </div>
                </section>

                <section class="filter-group">
                    <label class="filter-label">
                        Review count
                    </label>

                    <div class="input-pair">
                        <label class="input-wrapper">
                            <span class="input-caption">Minimum reviews</span>

                            <input
                                id="minReviews"
                                type="number"
                                min="20"
                                step="10"
                                value="20"
                            >
                        </label>

                        <label class="input-wrapper">
                            <span class="input-caption">Maximum reviews</span>

                            <input
                                id="maxReviews"
                                type="number"
                                min="20"
                                step="100"
                                value=""
                                placeholder="No limit"
                            >
                        </label>
                    </div>
                </section>

                <section class="filter-group">
                    <label class="filter-label">
                        Positive review score
                    </label>

                    <div class="input-pair">
                        <label class="input-wrapper">
                            <span class="input-caption">Minimum percentage</span>

                            <input
                                id="minReviewScore"
                                type="number"
                                min="0"
                                max="100"
                                step="1"
                                value="50"
                            >
                        </label>

                        <label class="input-wrapper">
                            <span class="input-caption">Maximum percentage</span>

                            <input
                                id="maxReviewScore"
                                type="number"
                                min="0"
                                max="100"
                                step="1"
                                value="100"
                            >
                        </label>
                    </div>

                    <span class="filter-help">
                        Positive reviews divided by total reviews.
                    </span>
                </section>

                <section class="filter-group">
                    <label
                        class="filter-label"
                        for="includeTags"
                    >
                        Must include tags
                    </label>

                    <input
                        id="includeTags"
                        type="text"
                        list="availableTags"
                        placeholder="Type or browse tags"
                    >

                    <datalist id="availableTags">
                        {tag_options_html}
                    </datalist>

                    <div
                        class="selected-tag-summary"
                        id="selectedTagSummary"
                    ></div>

                    <span class="filter-help">
                        Separate tags with commas. Every selected tag must
                        be present on the game.
                    </span>

                    <details class="tag-browser">
                        <summary>
                            Browse all {len(all_tags):,} tags
                        </summary>

                        <div class="tag-search-wrapper">
                            <input
                                id="tagBrowserSearch"
                                type="text"
                                placeholder="Search available tags"
                            >
                        </div>

                        <div
                            class="available-tag-list"
                            id="availableTagList"
                        >
                            {tag_buttons_html}
                        </div>
                    </details>
                </section>

                <section class="filter-group">
                    <label
                        class="filter-label"
                        for="excludeTags"
                    >
                        Exclude tags
                    </label>

                    <input
                        id="excludeTags"
                        type="text"
                        list="availableTags"
                        placeholder="e.g. Multiplayer, VR"
                    >
                </section>

                <section class="filter-group">
                    <div class="toggle-row">
                        <div class="toggle-copy">
                            <strong>Avoid repeats</strong>

                            <span>
                                Skip games already shown this session.
                            </span>
                        </div>

                        <label class="switch">
                            <input
                                id="avoidRepeats"
                                type="checkbox"
                                checked
                            >

                            <span class="switch-track"></span>
                        </label>
                    </div>
                </section>

                <div class="button-stack">
                    <button
                        class="primary-button"
                        id="suggestButton"
                        type="button"
                    >
                        Suggest a case study
                    </button>

                    <button
                        class="secondary-button"
                        id="resetButton"
                        type="button"
                    >
                        Reset filters
                    </button>
                </div>
            </aside>

            <section class="result-column">
                <div
                    class="panel empty-state"
                    id="emptyState"
                >
                    <div>
                        <div class="empty-icon">
                            ◈
                        </div>

                        <h2>
                            Your next case study is waiting
                        </h2>

                        <p>
                            The default view studies games earning an
                            estimated $150K–$800K. Narrow it further by
                            year, review score, price, rank, or Steam tags.
                        </p>
                    </div>
                </div>

                <article
                    class="panel case-study hidden"
                    id="caseStudy"
                >
                    <div class="hero">
                        <img
                            class="hero-image"
                            id="heroImage"
                            alt=""
                        >

                        <div class="hero-gradient"></div>

                        <div class="hero-content">
                            <div class="case-label">
                                Today's indie case study
                            </div>

                            <h2
                                class="game-title"
                                id="gameTitle"
                            ></h2>

                            <div
                                class="hero-meta"
                                id="heroMeta"
                            ></div>
                        </div>
                    </div>

                    <div class="case-content">
                        <div class="metric-grid">
                            <div class="metric">
                                <span class="metric-label">
                                    Estimated revenue
                                </span>

                                <span
                                    class="metric-value green"
                                    id="gameRevenue"
                                ></span>
                            </div>

                            <div class="metric">
                                <span class="metric-label">
                                    Listed price
                                </span>

                                <span
                                    class="metric-value"
                                    id="gamePrice"
                                ></span>
                            </div>

                            <div class="metric">
                                <span class="metric-label">
                                    Yearly rank
                                </span>

                                <span
                                    class="metric-value accent"
                                    id="gameRank"
                                ></span>
                            </div>

                            <div class="metric">
                                <span class="metric-label">
                                    Revenue percentile
                                </span>

                                <span
                                    class="metric-value"
                                    id="gamePercentile"
                                ></span>
                            </div>

                            <div class="metric">
                                <span class="metric-label">
                                    Steam reviews
                                </span>

                                <span
                                    class="metric-value"
                                    id="gameReviews"
                                ></span>
                            </div>

                            <div class="metric">
                                <span class="metric-label">
                                    Positive reviews
                                </span>

                                <span
                                    class="metric-value"
                                    id="gameReviewScore"
                                ></span>
                            </div>
                        </div>

                        <div class="details-grid">
                            <div>
                                <section>
                                    <h3 class="section-title">
                                        Store pitch
                                    </h3>

                                    <p
                                        class="description"
                                        id="gameDescription"
                                    ></p>
                                </section>

                                <section style="margin-top: 26px;">
                                    <h3 class="section-title">
                                        Steam tags
                                    </h3>

                                    <div
                                        class="tag-list"
                                        id="gameTags"
                                    ></div>
                                </section>
                            </div>

                            <aside>
                                <div class="rank-card">
                                    <span class="rank-card-label">
                                        Position among qualifying releases
                                    </span>

                                    <strong
                                        class="rank-card-value"
                                        id="rankCardValue"
                                    ></strong>

                                    <span
                                        class="rank-card-note"
                                        id="rankCardNote"
                                    ></span>
                                </div>
                            </aside>
                        </div>

                        <div class="actions">
                            <button
                                class="primary-button"
                                id="anotherButton"
                                type="button"
                            >
                                Another case study
                            </button>

                            <a
                                class="store-button"
                                id="steamLink"
                                target="_blank"
                                rel="noopener noreferrer"
                            >
                                Reveal Steam page ↗
                            </a>

                            <button
                                class="secondary-button"
                                id="copyAppIdButton"
                                type="button"
                            >
                                Copy AppID
                            </button>
                        </div>
                    </div>
                </article>

                <section
                    class="panel history-panel hidden"
                    id="historyPanel"
                >
                    <div class="panel-heading">
                        <h2>Session history</h2>

                        <button
                            class="ghost-button"
                            id="clearHistoryButton"
                            type="button"
                        >
                            Clear history
                        </button>
                    </div>

                    <div
                        class="history-list"
                        id="historyList"
                    ></div>
                </section>
            </section>
        </div>
    </main>

    <div
        class="toast"
        id="toast"
        role="status"
        aria-live="polite"
    ></div>

    <script id="game-data" type="application/json">
        {embedded_json}
    </script>

    <script>
        "use strict";

        const games = JSON.parse(
            document.getElementById("game-data").textContent
        );

        const elements = {{
            yearCheckboxes: [
                ...document.querySelectorAll(".year-checkbox")
            ],

            minRevenue:
                document.getElementById("minRevenue"),

            maxRevenue:
                document.getElementById("maxRevenue"),

            minPrice:
                document.getElementById("minPrice"),

            maxPrice:
                document.getElementById("maxPrice"),

            minRank:
                document.getElementById("minRank"),

            maxRank:
                document.getElementById("maxRank"),

            minPercentile:
                document.getElementById("minPercentile"),

            maxPercentile:
                document.getElementById("maxPercentile"),

            minReviews:
                document.getElementById("minReviews"),

            maxReviews:
                document.getElementById("maxReviews"),

            minReviewScore:
                document.getElementById("minReviewScore"),

            maxReviewScore:
                document.getElementById("maxReviewScore"),

            includeTags:
                document.getElementById("includeTags"),

            excludeTags:
                document.getElementById("excludeTags"),

            selectedTagSummary:
                document.getElementById("selectedTagSummary"),

            tagBrowserSearch:
                document.getElementById("tagBrowserSearch"),

            availableTagButtons: [
                ...document.querySelectorAll(".available-tag")
            ],

            avoidRepeats:
                document.getElementById("avoidRepeats"),

            matchCount:
                document.getElementById("matchCount"),

            suggestButton:
                document.getElementById("suggestButton"),

            resetButton:
                document.getElementById("resetButton"),

            emptyState:
                document.getElementById("emptyState"),

            caseStudy:
                document.getElementById("caseStudy"),

            heroImage:
                document.getElementById("heroImage"),

            gameTitle:
                document.getElementById("gameTitle"),

            heroMeta:
                document.getElementById("heroMeta"),

            gameRevenue:
                document.getElementById("gameRevenue"),

            gamePrice:
                document.getElementById("gamePrice"),

            gameRank:
                document.getElementById("gameRank"),

            gamePercentile:
                document.getElementById("gamePercentile"),

            gameReviews:
                document.getElementById("gameReviews"),

            gameReviewScore:
                document.getElementById("gameReviewScore"),

            gameDescription:
                document.getElementById("gameDescription"),

            gameTags:
                document.getElementById("gameTags"),

            rankCardValue:
                document.getElementById("rankCardValue"),

            rankCardNote:
                document.getElementById("rankCardNote"),

            anotherButton:
                document.getElementById("anotherButton"),

            steamLink:
                document.getElementById("steamLink"),

            copyAppIdButton:
                document.getElementById("copyAppIdButton"),

            historyPanel:
                document.getElementById("historyPanel"),

            historyList:
                document.getElementById("historyList"),

            clearHistoryButton:
                document.getElementById("clearHistoryButton"),
            gameLookup:
                document.getElementById("gameLookup"),

            loadGameButton:
                document.getElementById("loadGameButton"),
            toast:
                document.getElementById("toast")
        }};

        const state = {{
            currentGame: null,
            shownAppIds: new Set(),
            history: []
        }};

        function numericValue(element, fallback) {{
            if (
                element.value === ""
                || element.value === null
            ) {{
                return fallback;
            }}

            const value = Number(element.value);

            return Number.isFinite(value)
                ? value
                : fallback;
        }}

        function selectedYears() {{
            return new Set(
                elements.yearCheckboxes
                    .filter(checkbox => checkbox.checked)
                    .map(checkbox => Number(checkbox.value))
            );
        }}

        function parseTagInput(value) {{
            return value
                .split(",")
                .map(tag => tag.trim().toLocaleLowerCase())
                .filter(Boolean);
        }}

        function displayTagInput(value) {{
            return value
                .split(",")
                .map(tag => tag.trim())
                .filter(Boolean);
        }}

        function normalizedTags(game) {{
            if (Array.isArray(game.tags)) {{
                return game.tags
                    .map(tag => String(tag).trim())
                    .filter(Boolean);
            }}

            if (
                game.tags
                && typeof game.tags === "object"
            ) {{
                return Object.keys(game.tags);
            }}

            return [];
        }}

        function addIncludedTag(tag) {{
            const normalizedTag =
                String(tag).trim().toLocaleLowerCase();

            if (!normalizedTag) {{
                return;
            }}

            const existingNormalizedTags =
                parseTagInput(elements.includeTags.value);

            if (
                !existingNormalizedTags.includes(normalizedTag)
            ) {{
                const displayedTags =
                    displayTagInput(elements.includeTags.value);

                displayedTags.push(String(tag).trim());

                elements.includeTags.value =
                    displayedTags.join(", ");
            }}

            updateTagInterface();
            updateMatchCount();
        }}

        function removeIncludedTag(tagToRemove) {{
            const normalizedToRemove =
                String(tagToRemove)
                    .trim()
                    .toLocaleLowerCase();

            const remainingTags =
                displayTagInput(elements.includeTags.value)
                    .filter(
                        tag =>
                            tag.toLocaleLowerCase()
                            !== normalizedToRemove
                    );

            elements.includeTags.value =
                remainingTags.join(", ");

            updateTagInterface();
            updateMatchCount();
        }}

        function findGameByNameOrAppId(query) {{
            const cleanedQuery =
                String(query || "").trim();

            if (!cleanedQuery) {{
                return null;
            }}

            // First try an exact AppID
            if (/^\\d+$/.test(cleanedQuery)) {{
                const appid = Number(cleanedQuery);

                const appMatch = games.find(
                    game => Number(game.appid) === appid
                );

                if (appMatch) {{
                    return appMatch;
                }}
            }}

            const normalizedQuery =
                cleanedQuery.toLocaleLowerCase();

            // Then try an exact game-name match
            const exactNameMatch = games.find(
                game =>
                    String(game.name || "")
                        .trim()
                        .toLocaleLowerCase()
                    === normalizedQuery
            );

            if (exactNameMatch) {{
                return exactNameMatch;
            }}

            // Finally try a partial game-name match
            const partialMatches = games.filter(
                game =>
                    String(game.name || "")
                        .toLocaleLowerCase()
                        .includes(normalizedQuery)
            );

            if (partialMatches.length === 1) {{
                return partialMatches[0];
            }}

            if (partialMatches.length > 1) {{
                partialMatches.sort(
                    (a, b) =>
                        String(a.name).length
                        - String(b.name).length
                );

                return partialMatches[0];
            }}

            return null;
        }}

        function findGameByNameOrAppId(query) {{
            const cleanedQuery =
                String(query || "").trim();

            if (!cleanedQuery) {{
                return null;
            }}

            if (/^\\d+$/.test(cleanedQuery)) {{
                const appid = Number(cleanedQuery);

                const appMatch = games.find(
                    game => Number(game.appid) === appid
                );

                if (appMatch) {{
                    return appMatch;
                }}
            }}

            const normalizedQuery =
                cleanedQuery.toLocaleLowerCase();

            const exactNameMatch = games.find(
                game =>
                    String(game.name || "")
                        .trim()
                        .toLocaleLowerCase()
                    === normalizedQuery
            );

            if (exactNameMatch) {{
                return exactNameMatch;
            }}

            const partialMatches = games.filter(
                game =>
                    String(game.name || "")
                        .toLocaleLowerCase()
                        .includes(normalizedQuery)
            );

            if (partialMatches.length === 1) {{
                return partialMatches[0];
            }}

            if (partialMatches.length > 1) {{
                partialMatches.sort(
                    (a, b) =>
                        String(a.name).length
                        - String(b.name).length
                );

                return partialMatches[0];
            }}

            return null;
        }}


        function loadSpecificGame() {{
            const query =
                elements.gameLookup.value;

            const game =
                findGameByNameOrAppId(query);

            if (!game) {{
                showToast(
                    "No matching game found."
                );

                return;
            }}

            renderGame(game);

            elements.gameLookup.value =
                game.name;

            showToast(
                `Loaded ${{game.name}}.`
            );
        }}

        function loadSpecificGame() {{
            const query =
                elements.gameLookup.value;

            const game =
                findGameByNameOrAppId(query);

            if (!game) {{
                showToast(
                    "No matching game found."
                );

                return;
            }}

            renderGame(game);

            elements.gameLookup.value =
                game.name;

            showToast(
                `Loaded ${{game.name}}.`
            );
        }}


        function suggestGame() {{
        
        function updateTagInterface() {{
            const selectedTags =
                parseTagInput(elements.includeTags.value);

            elements.availableTagButtons.forEach(button => {{
                const tag =
                    button.dataset.tag
                        .trim()
                        .toLocaleLowerCase();

                button.classList.toggle(
                    "selected",
                    selectedTags.includes(tag)
                );
            }});

            const displayedTags =
                displayTagInput(elements.includeTags.value);

            elements.selectedTagSummary.innerHTML =
                displayedTags.map(tag => `
                    <span class="selected-filter-tag">
                        ${{escapeHtml(tag)}}

                        <button
                            type="button"
                            data-remove-tag="${{escapeHtml(tag)}}"
                            aria-label="Remove ${{escapeHtml(tag)}}"
                        >
                            ×
                        </button>
                    </span>
                `).join("");

            elements.selectedTagSummary
                .querySelectorAll("[data-remove-tag]")
                .forEach(button => {{
                    button.addEventListener(
                        "click",
                        () => {{
                            removeIncludedTag(
                                button.dataset.removeTag
                            );
                        }}
                    );
                }});
        }}

        function filterAvailableTagButtons() {{
            const query =
                elements.tagBrowserSearch.value
                    .trim()
                    .toLocaleLowerCase();

            elements.availableTagButtons.forEach(button => {{
                const tag =
                    button.dataset.tag
                        .toLocaleLowerCase();

                button.hidden =
                    query.length > 0
                    && !tag.includes(query);
            }});
        }}

        function filteredGames(
            {{ respectHistory = true }} = {{}}
        ) {{
            const years = selectedYears();

            const minRevenue =
                numericValue(elements.minRevenue, 0);

            const maxRevenue =
                numericValue(
                    elements.maxRevenue,
                    Number.POSITIVE_INFINITY
                );

            const minPrice =
                numericValue(elements.minPrice, 0);

            const maxPrice =
                numericValue(
                    elements.maxPrice,
                    Number.POSITIVE_INFINITY
                );

            const minRank =
                numericValue(elements.minRank, 1);

            const maxRank =
                numericValue(
                    elements.maxRank,
                    Number.POSITIVE_INFINITY
                );

            const minPercentile =
                numericValue(elements.minPercentile, 0);

            const maxPercentile =
                numericValue(elements.maxPercentile, 100);

            const minReviews =
                numericValue(elements.minReviews, 20);

            const maxReviews =
                numericValue(
                    elements.maxReviews,
                    Number.POSITIVE_INFINITY
                );

            const minReviewScore =
                numericValue(elements.minReviewScore, 0);

            const maxReviewScore =
                numericValue(elements.maxReviewScore, 100);

            const requiredTags =
                parseTagInput(elements.includeTags.value);

            const excludedTags =
                parseTagInput(elements.excludeTags.value);

            return games.filter(game => {{
                const appid = Number(game.appid);
                const gameYear = Number(game.release_year);

                const revenue = Number(
                    game.estimated_revenue || 0
                );

                const price = Number(
                    game.price || 0
                );

                const rank = Number(
                    game.revenue_rank_in_year || 0
                );

                const percentile = Number(
                    game.revenue_percentile_in_year || 0
                );

                const reviews = Number(
                    game.total_reviews || 0
                );

                const reviewScore = Number(
                    game.review_score || 0
                );

                const tags = normalizedTags(game)
                    .map(tag => tag.toLocaleLowerCase());

                const includesEveryRequiredTag =
                    requiredTags.every(requiredTag =>
                        tags.some(tag =>
                            tag === requiredTag
                            || tag.includes(requiredTag)
                        )
                    );

                const includesExcludedTag =
                    excludedTags.some(excludedTag =>
                        tags.some(tag =>
                            tag === excludedTag
                            || tag.includes(excludedTag)
                        )
                    );

                const isRepeated =
                    respectHistory
                    && elements.avoidRepeats.checked
                    && state.shownAppIds.has(appid);

                return (
                    years.has(gameYear)
                    && revenue >= minRevenue
                    && revenue <= maxRevenue
                    && price >= minPrice
                    && price <= maxPrice
                    && rank >= minRank
                    && rank <= maxRank
                    && percentile >= minPercentile
                    && percentile <= maxPercentile
                    && reviews >= minReviews
                    && reviews <= maxReviews
                    && reviewScore >= minReviewScore
                    && reviewScore <= maxReviewScore
                    && includesEveryRequiredTag
                    && !includesExcludedTag
                    && !isRepeated
                );
            }});
        }}

        function updateMatchCount() {{
            let matches = filteredGames();

            if (
                matches.length === 0
                && elements.avoidRepeats.checked
            ) {{
                const withoutHistory = filteredGames({{
                    respectHistory: false
                }});

                if (withoutHistory.length > 0) {{
                    elements.matchCount.textContent =
                        "All matches seen";

                    elements.suggestButton.disabled = false;
                    return;
                }}
            }}

            elements.matchCount.textContent =
                `${{matches.length.toLocaleString()}} matches`;

            elements.suggestButton.disabled =
                matches.length === 0;
        }}

        function formatCurrency(value) {{
            return new Intl.NumberFormat(
                "en-US",
                {{
                    style: "currency",
                    currency: "USD",
                    maximumFractionDigits:
                        Number(value) >= 1000 ? 0 : 2
                }}
            ).format(Number(value || 0));
        }}

        function formatCompactCurrency(value) {{
            const number = Number(value || 0);

            if (number >= 1_000_000_000) {{
                return `$${{
                    (number / 1_000_000_000).toFixed(1)
                }}B`;
            }}

            if (number >= 1_000_000) {{
                return `$${{
                    (number / 1_000_000).toFixed(1)
                }}M`;
            }}

            if (number >= 1_000) {{
                return `$${{
                    Math.round(number / 1_000)
                }}K`;
            }}

            return formatCurrency(number);
        }}

        function escapeHtml(value) {{
            const container = document.createElement("div");
            container.textContent = String(value ?? "");
            return container.innerHTML;
        }}

        function marketingImageUrls(appid) {{
            const base =
                `https://shared.akamai.steamstatic.com/`
                + `store_item_assets/steam/apps/${{appid}}/`;

            return [
                `${{base}}header.jpg`,
                `${{base}}capsule_616x353.jpg`,
                `https://cdn.akamai.steamstatic.com/`
                    + `steam/apps/${{appid}}/header.jpg`
            ];
        }}

        function loadMarketingImage(game) {{
            const urls = marketingImageUrls(game.appid);
            let urlIndex = 0;

            elements.heroImage.classList.add("loading");

            elements.heroImage.alt =
                `${{game.name}} Steam store artwork`;

            function tryNextImage() {{
                if (urlIndex >= urls.length) {{
                    elements.heroImage.onerror = null;
                    elements.heroImage.onload = null;

                    elements.heroImage.src =
                        "data:image/svg+xml;charset=UTF-8,"
                        + encodeURIComponent(`
                            <svg
                                xmlns="http://www.w3.org/2000/svg"
                                width="1200"
                                height="600"
                                viewBox="0 0 1200 600"
                            >
                                <rect
                                    width="1200"
                                    height="600"
                                    fill="#0d1420"
                                />

                                <text
                                    x="600"
                                    y="285"
                                    fill="#7f91aa"
                                    font-family="Arial, sans-serif"
                                    font-size="36"
                                    text-anchor="middle"
                                >
                                    Steam artwork unavailable
                                </text>

                                <text
                                    x="600"
                                    y="345"
                                    fill="#53647c"
                                    font-family="Arial, sans-serif"
                                    font-size="25"
                                    text-anchor="middle"
                                >
                                    AppID ${{game.appid}}
                                </text>
                            </svg>
                        `);

                    elements.heroImage.classList.remove(
                        "loading"
                    );

                    return;
                }}

                elements.heroImage.src =
                    urls[urlIndex];

                urlIndex += 1;
            }}

            elements.heroImage.onload = () => {{
                elements.heroImage.classList.remove(
                    "loading"
                );
            }};

            elements.heroImage.onerror = tryNextImage;

            tryNextImage();
        }}

        function renderGame(game) {{
            state.currentGame = game;

            const tags = normalizedTags(game);

            elements.gameTitle.textContent =
                game.name || `AppID ${{game.appid}}`;

            const heroChips = [
                `<span class="hero-chip">${{
                    escapeHtml(game.release_year)
                }}</span>`,

                `<span class="hero-chip">AppID ${{
                    escapeHtml(game.appid)
                }}</span>`
            ];

            if (tags.length > 0) {{
                heroChips.push(
                    `<span class="hero-chip">${{
                        escapeHtml(
                            tags.slice(0, 3).join(" · ")
                        )
                    }}</span>`
                );
            }}

            elements.heroMeta.innerHTML =
                heroChips.join("");

            elements.gameRevenue.textContent =
                formatCompactCurrency(
                    game.estimated_revenue
                );

            elements.gamePrice.textContent =
                Number(game.price || 0) === 0
                    ? "Free"
                    : formatCurrency(game.price);

            elements.gameRank.textContent =
                `#${{
                    Number(
                        game.revenue_rank_in_year
                    ).toLocaleString()
                }}`;

            elements.gamePercentile.textContent =
                `${{
                    Number(
                        game.revenue_percentile_in_year || 0
                    ).toFixed(1)
                }}%`;

            elements.gameReviews.textContent =
                Number(
                    game.total_reviews || 0
                ).toLocaleString();

            elements.gameReviewScore.textContent =
                `${{
                    Number(
                        game.review_score || 0
                    ).toFixed(1)
                }}%`;

            elements.gameDescription.textContent =
                game.short_description
                || "No short store description is available.";

            elements.gameTags.innerHTML = tags.length
                ? tags.map(tag =>
                    `<span class="tag">${{
                        escapeHtml(tag)
                    }}</span>`
                ).join("")
                : `<span class="tag">No tags available</span>`;

            const rank = Number(
                game.revenue_rank_in_year || 0
            );

            const yearTotal = Number(
                game.qualifying_games_in_year || 0
            );

            const percentile = Number(
                game.revenue_percentile_in_year || 0
            );

            const topShare = Math.max(
                0,
                100 - percentile
            );

            elements.rankCardValue.textContent =
                `#${{rank.toLocaleString()}} of `
                + `${{yearTotal.toLocaleString()}}`;

            elements.rankCardNote.textContent =
                `Approximately the top `
                + `${{topShare.toFixed(1)}}% of qualifying `
                + `Steam releases from ${{game.release_year}}.`;

            elements.steamLink.href =
                `https://store.steampowered.com/app/${{
                    game.appid
                }}`;

            loadMarketingImage(game);

            elements.emptyState.classList.add(
                "hidden"
            );

            elements.caseStudy.classList.remove(
                "hidden"
            );

            addToHistory(game);
        }}

        function addToHistory(game) {{
            const appid = Number(game.appid);

            state.shownAppIds.add(appid);

            state.history = [
                game,
                ...state.history.filter(
                    existing =>
                        Number(existing.appid) !== appid
                )
            ].slice(0, 12);

            renderHistory();
            updateMatchCount();
        }}

        function renderHistory() {{
            if (state.history.length === 0) {{
                elements.historyPanel.classList.add(
                    "hidden"
                );

                elements.historyList.innerHTML = "";
                return;
            }}

            elements.historyPanel.classList.remove(
                "hidden"
            );

            elements.historyList.innerHTML =
                state.history.map((game, index) => `
                    <div class="history-item">
                        <span class="history-rank">
                            ${{index + 1}}
                        </span>

                        <span class="history-name">
                            ${{escapeHtml(game.name)}}
                        </span>

                        <span class="history-revenue">
                            ${{formatCompactCurrency(
                                game.estimated_revenue
                            )}}
                        </span>
                    </div>
                `).join("");
        }}

        function suggestGame() {{
            let candidates = filteredGames();

            if (
                candidates.length === 0
                && elements.avoidRepeats.checked
            ) {{
                const candidatesIgnoringHistory =
                    filteredGames({{
                        respectHistory: false
                    }});

                if (
                    candidatesIgnoringHistory.length > 0
                ) {{
                    state.shownAppIds.clear();

                    candidates =
                        candidatesIgnoringHistory;

                    showToast(
                        "All matching games had been shown. "
                        + "Repeat history was reset."
                    );
                }}
            }}

            if (candidates.length === 0) {{
                showToast(
                    "No games match these filters."
                );

                updateMatchCount();
                return;
            }}

            const selected =
                candidates[
                    Math.floor(
                        Math.random() * candidates.length
                    )
                ];

            renderGame(selected);
        }}

        function resetFilters() {{
            elements.yearCheckboxes.forEach(
                checkbox => {{
                    checkbox.checked = true;
                }}
            );

            elements.minRevenue.value = 150000;
            elements.maxRevenue.value = 800000;

            elements.minPrice.value = 0;
            elements.maxPrice.value =
                {max(100, int(maximum_price) + 1)};

            elements.minRank.value = 1;
            elements.maxRank.value = {maximum_rank};

            elements.minPercentile.value = 0;
            elements.maxPercentile.value = 100;

            elements.minReviews.value = 20;
            elements.maxReviews.value = "";

            elements.minReviewScore.value = 50;
            elements.maxReviewScore.value = 100;

            elements.includeTags.value = "";
            elements.excludeTags.value = "";

            elements.tagBrowserSearch.value = "";

            elements.avoidRepeats.checked = true;

            filterAvailableTagButtons();
            updateTagInterface();
            updateMatchCount();

            showToast("Filters reset.");
        }}

        async function copyCurrentAppId() {{
            if (!state.currentGame) {{
                return;
            }}

            const appid = String(
                state.currentGame.appid
            );

            try {{
                await navigator.clipboard.writeText(
                    appid
                );

                showToast(
                    `Copied AppID ${{appid}}.`
                );
            }}
            catch {{
                const temporaryInput =
                    document.createElement("textarea");

                temporaryInput.value = appid;

                document.body.appendChild(
                    temporaryInput
                );

                temporaryInput.select();

                document.execCommand("copy");

                temporaryInput.remove();

                showToast(
                    `Copied AppID ${{appid}}.`
                );
            }}
        }}

        let toastTimer = null;

        function showToast(message) {{
            elements.toast.textContent = message;

            elements.toast.classList.add(
                "visible"
            );

            window.clearTimeout(toastTimer);

            toastTimer = window.setTimeout(
                () => {{
                    elements.toast.classList.remove(
                        "visible"
                    );
                }},
                2400
            );
        }}

        const filterInputs = [
            ...elements.yearCheckboxes,
            elements.minRevenue,
            elements.maxRevenue,
            elements.minPrice,
            elements.maxPrice,
            elements.minRank,
            elements.maxRank,
            elements.minPercentile,
            elements.maxPercentile,
            elements.minReviews,
            elements.maxReviews,
            elements.minReviewScore,
            elements.maxReviewScore,
            elements.includeTags,
            elements.excludeTags,
            elements.avoidRepeats
        ];

        filterInputs.forEach(element => {{
            element.addEventListener(
                "input",
                () => {{
                    if (
                        element === elements.includeTags
                    ) {{
                        updateTagInterface();
                    }}

                    updateMatchCount();
                }}
            );

            element.addEventListener(
                "change",
                () => {{
                    if (
                        element === elements.includeTags
                    ) {{
                        updateTagInterface();
                    }}

                    updateMatchCount();
                }}
            );
        }});

        elements.availableTagButtons.forEach(
            button => {{
                button.addEventListener(
                    "click",
                    () => {{
                        addIncludedTag(
                            button.dataset.tag
                        );
                    }}
                );
            }}
        );

        elements.tagBrowserSearch.addEventListener(
            "input",
            filterAvailableTagButtons
        );

        elements.suggestButton.addEventListener(
            "click",
            suggestGame
        );

        elements.anotherButton.addEventListener(
            "click",
            suggestGame
        );

        elements.resetButton.addEventListener(
            "click",
            resetFilters
        );

        elements.copyAppIdButton.addEventListener(
            "click",
            copyCurrentAppId
        );

        elements.clearHistoryButton.addEventListener(
            "click",
            () => {{
                state.shownAppIds.clear();
                state.history = [];

                renderHistory();
                updateMatchCount();

                showToast(
                    "Session history cleared."
                );
            }}
        );

        
        elements.loadGameButton.addEventListener(
            "click",
            loadSpecificGame
        );

        elements.gameLookup.addEventListener(
            "keydown",
            event => {{
                if (event.key === "Enter") {{
                    event.preventDefault();
                    loadSpecificGame();
                }}
            }}
        );    

        document.addEventListener(
            "keydown",
            event => {{
                if (event.key !== "Enter") {{
                    return;
                }}

                if (
                    document.activeElement
                    === elements.gameLookup
                ) {{
                    return;
                }}

                if (
                    document.activeElement?.tagName
                    !== "BUTTON"
                ) {{
                    suggestGame();
                }}
            }}
        );

        filterAvailableTagButtons();
        updateTagInterface();
        updateMatchCount();
    </script>
</body>
</html>
"""

# ============================================================
# WRITE THE APP
# ============================================================

APP_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

html_bytes = html_document.encode(
    "utf-8",
    errors="replace"
)

HTML_OUTPUT_FILE.write_bytes(
    html_bytes
)

file_size_mb = (
    HTML_OUTPUT_FILE.stat().st_size
    / 1_000_000
)

print("Created Steam suggester:")
print(HTML_OUTPUT_FILE.resolve())
print()
print(f"Embedded games: {len(games_data):,}")
print(f"Available tags: {len(all_tags):,}")
print(f"File size: {file_size_mb:.2f} MB")

Created Steam suggester:
D:\Workstation\python\steam-analysis\steam-game-suggester\steam_game_suggester.html

Embedded games: 15,521
Available tags: 443
File size: 11.18 MB


In [3]:
print("PROJECT_ROOT:", PROJECT_ROOT)
print("APP_OUTPUT_DIR:", APP_OUTPUT_DIR)
print("HTML_OUTPUT_FILE:", HTML_OUTPUT_FILE)
print("Resolved:", HTML_OUTPUT_FILE.resolve())
print("Exists:", APP_OUTPUT_DIR.exists())
print("HTML characters:", len(html_document))

PROJECT_ROOT: d:\Workstation\python\steam-analysis
APP_OUTPUT_DIR: d:\Workstation\python\steam-analysis\steam-game-suggester
HTML_OUTPUT_FILE: d:\Workstation\python\steam-analysis\steam-game-suggester\steam_game_suggester.html
Resolved: D:\Workstation\python\steam-analysis\steam-game-suggester\steam_game_suggester.html
Exists: True
HTML characters: 11092178
